# Road Model Dataset v2 Information

This notebook inspects `data/road_model_dataset_v2.parquet` with a focus on TP-derived features:

- `tp_surface_type`
- `tp_material_type`
- `has_TP_interval`
- `tp_count_interval`

Main questions:

- How often do we know the pavement surface category?
- How often do we know the treatment material / work category?
- What are the category frequencies and distributions?
- How do these features vary by year and by measurement position in the lifecycle?
- Are there edge cases where TP was observed but the simplified category is still `none`?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("ggplot")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
DATA_PATH = Path("data/road_model_dataset_v2.parquet")

COLUMNS = [
    "Segment_ID",
    "Lifecycle_ID",
    "event_date",
    "year",
    "Measurement_Idx",
    "Pavement_Age_years",
    "target_horizon_years",
    "has_TP_interval",
    "tp_count_interval",
    "tp_surface_type",
    "tp_material_type",
]

df = pd.read_parquet(DATA_PATH, columns=COLUMNS).copy()
df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
if "year" not in df.columns or df["year"].isna().all():
    df["year"] = df["event_date"].dt.year

df["surface_known"] = df["tp_surface_type"].fillna("none").ne("none")
df["material_known"] = df["tp_material_type"].fillna("none").ne("none")
df["both_known"] = df["surface_known"] & df["material_known"]

print(f"Rows: {len(df):,}")
print(f"Segments: {df['Segment_ID'].nunique():,}")
print(f"Lifecycles: {df['Lifecycle_ID'].nunique():,}")
print(f"Date range: {df['event_date'].min().date()} -> {df['event_date'].max().date()}")

## Overall Coverage

These summaries show how often TP-derived surface and material information is known versus `none`.

In [ ]:
coverage = pd.DataFrame(
    {
        "metric": [
            "surface_known_share",
            "material_known_share",
            "both_known_share",
            "has_TP_interval_share",
            "mean_tp_count_interval",
            "median_tp_count_interval",
        ],
        "value": [
            df["surface_known"].mean(),
            df["material_known"].mean(),
            df["both_known"].mean(),
            df["has_TP_interval"].fillna(False).mean(),
            df["tp_count_interval"].fillna(0).mean(),
            df["tp_count_interval"].fillna(0).median(),
        ],
    }
)
coverage

In [ ]:
surface_counts = df["tp_surface_type"].fillna("missing").value_counts(dropna=False)
surface_share = df["tp_surface_type"].fillna("missing").value_counts(normalize=True, dropna=False)
surface_summary = pd.DataFrame({"count": surface_counts, "share": surface_share})
surface_summary

In [ ]:
material_counts = df["tp_material_type"].fillna("missing").value_counts(dropna=False)
material_share = df["tp_material_type"].fillna("missing").value_counts(normalize=True, dropna=False)
material_summary = pd.DataFrame({"count": material_counts, "share": material_share})
material_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

surface_summary["count"].plot(kind="bar", ax=axes[0], color="#1f77b4")
axes[0].set_title("tp_surface_type frequency")
axes[0].set_xlabel("")
axes[0].set_ylabel("Rows")

material_summary["count"].plot(kind="bar", ax=axes[1], color="#ff7f0e")
axes[1].set_title("tp_material_type frequency")
axes[1].set_xlabel("")
axes[1].set_ylabel("Rows")

plt.tight_layout()

## Joint Distribution

This cross-tab shows which surface and material categories appear together.

In [ ]:
joint_counts = pd.crosstab(df["tp_surface_type"], df["tp_material_type"], margins=True)
joint_counts

In [ ]:
joint_share = pd.crosstab(
    df["tp_surface_type"],
    df["tp_material_type"],
    normalize="all",
)
joint_share

## Coverage by Year

The early years are especially interesting because the event history is truncated on the left side.

In [ ]:
coverage_by_year = (
    df.groupby("year", dropna=False)
    .agg(
        rows=("Segment_ID", "size"),
        has_TP_interval_share=("has_TP_interval", lambda s: s.fillna(False).mean()),
        surface_known_share=("surface_known", "mean"),
        material_known_share=("material_known", "mean"),
        mean_tp_count_interval=("tp_count_interval", lambda s: s.fillna(0).mean()),
    )
)
coverage_by_year

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
coverage_by_year[["surface_known_share", "material_known_share", "has_TP_interval_share"]].plot(ax=ax)
ax.set_title("Known TP-derived information by year")
ax.set_ylabel("Share of rows")
ax.set_xlabel("Year")
plt.tight_layout()

## Coverage by Measurement Position

This shows whether TP-derived information is concentrated in early or later measurements within a lifecycle.

In [ ]:
coverage_by_measurement = (
    df.groupby("Measurement_Idx", dropna=False)
    .agg(
        rows=("Segment_ID", "size"),
        surface_known_share=("surface_known", "mean"),
        material_known_share=("material_known", "mean"),
        has_TP_interval_share=("has_TP_interval", lambda s: s.fillna(False).mean()),
        mean_tp_count_interval=("tp_count_interval", lambda s: s.fillna(0).mean()),
    )
    .sort_index()
)

coverage_by_measurement.head(20)

In [ ]:
plot_measurement = coverage_by_measurement[coverage_by_measurement["rows"] >= 1000].copy()

fig, ax = plt.subplots(figsize=(12, 5))
plot_measurement[["surface_known_share", "material_known_share", "has_TP_interval_share"]].plot(ax=ax)
ax.set_title("Known TP-derived information by Measurement_Idx")
ax.set_ylabel("Share of rows")
ax.set_xlabel("Measurement_Idx")
plt.tight_layout()

## Interesting Edge Cases

These checks look for rows where `has_TP_interval == True` but the simplified category is still `none`. That can happen when the raw TP metadata itself is missing.

In [ ]:
edge_case_summary = pd.DataFrame(
    {
        "metric": [
            "surface_none_despite_tp_interval",
            "material_none_despite_tp_interval",
            "surface_other",
            "material_other",
        ],
        "count": [
            ((df["has_TP_interval"] == True) & df["tp_surface_type"].eq("none")).sum(),
            ((df["has_TP_interval"] == True) & df["tp_material_type"].eq("none")).sum(),
            df["tp_surface_type"].eq("other").sum(),
            df["tp_material_type"].eq("other").sum(),
        ],
    }
)
edge_case_summary

In [ ]:
df[
    (df["has_TP_interval"] == True)
    & (
        df["tp_surface_type"].eq("none")
        | df["tp_material_type"].eq("none")
        | df["tp_surface_type"].eq("other")
        | df["tp_material_type"].eq("other")
    )
].head(20)

## TP Count Distribution

A compact look at how many TP events are recorded in the PTM interval.

In [ ]:
tp_count_summary = df["tp_count_interval"].fillna(0).value_counts().sort_index()
tp_count_summary.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
tp_count_summary.loc[tp_count_summary.index <= 10].plot(kind="bar", ax=ax, color="#2ca02c")
ax.set_title("tp_count_interval distribution for values 0-10")
ax.set_xlabel("tp_count_interval")
ax.set_ylabel("Rows")
plt.tight_layout()

## Optional: First Measurement Focus

The first measurement in a lifecycle often behaves differently from later ones, so this is useful when thinking about cutoff effects and initialization.

In [ ]:
first_measurement = df[df["Measurement_Idx"] == 1].copy()
later_measurements = df[df["Measurement_Idx"] > 1].copy()

comparison = pd.DataFrame(
    {
        "subset": ["Measurement_Idx == 1", "Measurement_Idx > 1"],
        "rows": [len(first_measurement), len(later_measurements)],
        "surface_known_share": [first_measurement["surface_known"].mean(), later_measurements["surface_known"].mean()],
        "material_known_share": [first_measurement["material_known"].mean(), later_measurements["material_known"].mean()],
        "has_TP_interval_share": [
            first_measurement["has_TP_interval"].fillna(False).mean(),
            later_measurements["has_TP_interval"].fillna(False).mean(),
        ],
    }
)
comparison